# Course 7: Privacy-Preserving Machine Learning
## Session 11: Fairness, privacy and synthetic data

**Goal:** measure accuracy by subgroup at several values of epsilon, and build a small PATE teacher ensemble.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Data with a small subgroup
Two groups of patients. The minority group is 8% of the data and follows a different rule. To do well on it, the model must learn from few examples, which is exactly what clipping and noise make hard.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from math import comb, exp, log, sqrt
from torch.func import functional_call, vmap, grad

# ---- privacy accounting (Renyi DP of the sampled Gaussian mechanism, whole-number orders) ----
def rdp(q, sigma, alpha):
    """Renyi DP at order alpha of one DP-SGD step. q = sampling rate, sigma = noise multiplier."""
    if q == 1.0: return alpha / (2 * sigma ** 2)
    terms = [log(comb(alpha, k)) + (alpha - k) * log(1 - q) + k * log(q) + (k * k - k) / (2 * sigma ** 2) for k in range(alpha + 1)]
    m = max(terms)
    return (m + log(sum(exp(t - m) for t in terms))) / (alpha - 1)

def epsilon(q, sigma, steps, delta):
    """Convert composed Renyi DP to (epsilon, delta)-DP and keep the best order."""
    return min(steps * rdp(q, sigma, a) + log(1 / delta) / (a - 1) for a in range(2, 129))

# ---- DP-SGD ----
def dp_sgd(model, X, y, sigma, clip=1.0, lr=0.5, steps=300, q=0.1, seed=0, loss_fn=F.cross_entropy):
    """Poisson sampling, per-example clipping, Gaussian noise. sigma=0 with clip=inf is ordinary SGD."""
    g = torch.Generator().manual_seed(seed)
    params = {k: v.detach().clone() for k, v in model.named_parameters()}
    one = lambda p, x, t: loss_fn(functional_call(model, p, (x[None],)), t[None])
    per_example_grads = vmap(grad(one), in_dims=(None, 0, 0))
    n = len(X)
    for _ in range(steps):
        idx = torch.nonzero(torch.rand(n, generator=g) < q).squeeze(-1)
        grads = per_example_grads(params, X[idx], y[idx])
        norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in grads.values()))
        scale = (clip / (norms + 1e-12)).clamp(max=1.0)                     # shrink any gradient longer than clip
        for k in params:
            total = (grads[k] * scale.view(-1, *[1] * (grads[k].dim() - 1))).sum(0)
            if sigma > 0: total = total + torch.randn(total.shape, generator=g) * sigma * clip
            params[k] -= lr * total / (q * n)
    model.load_state_dict(params)
    return model

def mlp(d_in, n_out, hidden=32):
    return nn.Sequential(nn.Linear(d_in, hidden), nn.Tanh(), nn.Linear(hidden, n_out))

def accuracy(model, X, y):
    with torch.no_grad(): return (model(X).argmax(1) == y).float().mean().item()

In [ ]:
rng = np.random.default_rng(0); torch.manual_seed(0)
def make(n, minority_share=0.08):
    g = rng.random(n) < minority_share
    X = rng.normal(size=(n, 10))
    w_major, w_minor = np.array([2, -1, 1, 0, 0, 0, 0, 0, 0, 0.]), np.array([0, 0, 0, 2, -1, 1, 0, 0, 0, 0.])
    y = np.where(g, X @ w_minor > 0, X @ w_major > 0).astype(int)
    X = np.c_[X, g, X * g[:, None]]                                   # the group flag and its interactions are available
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y), g
Xtr, ytr, gtr = make(3000); Xte, yte, gte = make(6000)
print("training examples: majority", int((~gtr).sum()), "| minority", int(gtr.sum()))

### 2. Accuracy by subgroup against epsilon

In [ ]:
def by_group(model):
    with torch.no_grad(): ok = (model(Xte).argmax(1) == yte).numpy()
    return ok[~gte].mean(), ok[gte].mean()

STEPS, Q, DELTA = 400, 0.1, 1e-4
rows = []
for sigma in [None, 0.6, 1.0, 2.0, 4.0, 8.0]:
    res = []
    for seed in range(3):
        torch.manual_seed(seed)
        m = dp_sgd(mlp(21, 2), Xtr, ytr, sigma=sigma or 0.0, clip=1.0 if sigma else float("inf"), lr=1.0 if sigma else 0.3, steps=STEPS, q=Q, seed=seed)
        res.append(by_group(m))
    maj, mino = np.mean(res, axis=0)
    eps = float("inf") if sigma is None else epsilon(Q, sigma, STEPS, DELTA)
    rows.append((eps, maj, mino)); print(f"epsilon {eps:7.2f}   majority accuracy {maj:.3f}   minority accuracy {mino:.3f}   gap {maj - mino:+.3f}")
assert rows[-1][1] - rows[-1][2] > rows[0][1] - rows[0][2]

The cost of privacy is not shared equally: as epsilon shrinks, the minority group loses accuracy first and loses more. This is the finding of Bagdasaryan et al. (2019). In a medical setting, check every subgroup you care about before quoting one overall number.

### 3. PATE: private labels from an ensemble of teachers
Split the private data into disjoint parts and train one teacher on each. To label a public example, the teachers vote, noise is added to the vote counts, and only the winning label is released. One patient can change one teacher's vote, so the counts have low sensitivity.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
d = load_digits(); perm = rng.permutation(len(d.data))
private, public, test = perm[:1200], perm[1200:1500], perm[1500:]
N_TEACHERS = 30
teachers = [LogisticRegression(max_iter=300).fit(d.data[p] / 16, d.target[p]) for p in np.array_split(private, N_TEACHERS)]
votes = np.stack([t.predict(d.data[public] / 16) for t in teachers])                 # (teachers, public examples)
counts = np.stack([(votes == c).sum(0) for c in range(10)], 1)

def noisy_labels(gamma, seed=0):
    return (counts + np.random.default_rng(seed).laplace(scale=1 / gamma, size=counts.shape)).argmax(1)

for gamma in [0.02, 0.05, 0.2, 1.0]:
    lab = noisy_labels(gamma)
    student = LogisticRegression(max_iter=300).fit(d.data[public] / 16, lab)
    print(f"noise scale 1/{gamma:<5} per-query epsilon {2 * gamma:4.2f}   label accuracy {(lab == d.target[public]).mean():.3f}   student test accuracy {student.score(d.data[test] / 16, d.target[test]):.3f}")

Each answered query costs `2 * gamma` under the simple analysis, and 300 queries add up, so the PATE paper uses a data-dependent accountant: when the teachers agree strongly, a query costs far less. The student is what gets released; the teachers stay private.

### 4. Synthetic data is not automatically private
A generator trained without DP can memorize and replay its training rows. Private synthetic data means the generator itself was trained with DP (for example with DP-SGD), and then the fairness question above applies to it too: small groups are the first to be distorted.

### TODO
1. Change the minority share to 2% and to 20%. How does the gap at epsilon near 1 change?
2. Try two fixes: sample minority examples more often, or use a larger clip. Does either one close the gap, and what does it do to the accounting?
3. Count how many queries PATE can answer within a total epsilon of 5 using simple composition at gamma = 0.05.